# P5 — Putting it together

Three models, one long-range retrieval task: **GAT** (1-hop, learned), the **walk operator** (multi-hop,
fixed weights), and **Walk Attention** (multi-hop, learned). The theory predicts the outcome; the code
confirms it.

## 1. Three operators on the same graph

<img src="../figs-theory/en/T8_three_operators.svg" width="760"/>

## 2. Two axes: support and weights

<img src="../figs-theory/en/E5a_support_weights_table.svg" width="760"/>

## 3. Why each model wins or loses

<img src="../figs-theory/en/E5b_why_each.svg" width="760"/>

In [ ]:
# Train the three models on the bottleneck retrieval task (small, fast version).
import torch, torch.nn.functional as F, numpy as np, math
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGLoader
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.transforms import (AttachWalkMasks, collate_walk_masks,
                                   AttachWalkOperators, collate_walk_operators)
from oversquash.models import build_model
K, M, DEPTH = 5, 4, 3   # chance = 1/5 = 0.20
def ds(seed, n=1500):
    g = torch.Generator().manual_seed(seed)
    return [make_bottleneck_graph(K, M, DEPTH, g) for _ in range(n)]

def train_eval(m, tr, va, fwd, epochs=70, lr=2e-3, warmup=8):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: (e+1)/warmup if e<warmup
                                            else 0.5*(1+math.cos(math.pi*(e-warmup)/max(1,epochs-warmup))))
    best = 0.0
    for e in range(epochs):
        m.train()
        for b in tr:
            opt.zero_grad(); lg,_ = fwd(m,b)
            F.cross_entropy(lg, b.y, ignore_index=-100).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step(); m.eval(); a=[]
        with torch.no_grad():
            for b in va:
                lg,_ = fwd(m,b); mm=b.root_mask
                a.append((lg[mm].argmax(-1)==b.y[mm]).float().mean().item())
        best = max(best, float(np.mean(a)))
    return best

def run(name, seed=0):
    torch.manual_seed(seed); data = ds(seed); nl = DEPTH+1
    if name=='gat':
        tr=PyGLoader(data[:1200],batch_size=128,shuffle=True); va=PyGLoader(data[1200:],batch_size=128)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None))
    elif name=='walkraw':
        tf=AttachWalkOperators(n_layers=nl); data=[tf(d) for d in data]
        tr=DataLoader(data[:1200],batch_size=128,shuffle=True,collate_fn=collate_walk_operators)
        va=DataLoader(data[1200:],batch_size=128,collate_fn=collate_walk_operators)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None),walk_raw=b.walk_raw)
    else:
        tf=AttachWalkMasks(n_layers=nl); data=[tf(d) for d in data]
        tr=DataLoader(data[:1200],batch_size=128,shuffle=True,collate_fn=collate_walk_masks)
        va=DataLoader(data[1200:],batch_size=128,collate_fn=collate_walk_masks)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None),walk_masks=b.walk_masks)
    m = build_model(name, data[0].x.size(-1), 16, data[0].num_classes, nl, heads=4)
    return train_eval(m, tr, va, fwd)

for name in ['gat','walkraw','walkattn']:
    print(f'{name:10s} final accuracy = {run(name):.3f}')

## 4. The result

GAT collapses to chance; the walk operator reaches but cannot select; Walk Attention solves it (1.000 over 5
seeds in the full run).

<img src="../figs-theory/en/E5c_result.svg" width="760"/>

**The path algebra of a quiver defines a sparse multi-hop attention that mitigates over-squashing.**